# Task 1
**Understand Language Models & the SLM Concept**

What goes into the model and what comes out??
Ans- Basically we are going to input and sentence of tokens and our model is gonna predict the next token. It does this by first learning on huge datasets, of sentences then predicts the token, checks with the actual token and then updates its weight, accordingly.

In [23]:
from datasets import load_dataset
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [24]:
datset= load_dataset("roneneldan/TinyStories")

In [25]:
print(datset)
print(datset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


# Task 2
**Tokenize the Dataset & Save to Binary**

We tokenize because our neural network cannot understand text , it only understands number.

In [26]:
import tiktoken

encoder= tiktoken.get_encoding("gpt2")

text= "Hello my name is Anubhav Chatterjee."

tokens= encoder.encode(text)

print(tokens)

print(encoder.decode(tokens))


[15496, 616, 1438, 318, 1052, 549, 71, 615, 609, 1436, 34589, 13]
Hello my name is Anubhav Chatterjee.


In [27]:
def tokenize(sample):
    return {
        "input_ids": encoder.encode(sample["text"]),
    }

In [28]:
tokenized_dataset= datset.map(tokenize)

In [ ]:
#print(tokenized_dataset)



train_ids = np.concatenate(tokenized_dataset["train"]["input_ids"]).astype(np.uint16)
val_ids = np.concatenate(tokenized_dataset["validation"]["input_ids"]).astype(np.uint16)

train_bin = np.memmap("train.bin", dtype=np.uint16, mode="w+", shape=(len(train_ids),))
train_bin[:] = train_ids[:]
train_bin.flush()

val_bin = np.memmap("val.bin", dtype=np.uint16, mode="w+", shape=(len(val_ids),))
val_bin[:] = val_ids[:]
val_bin.flush()

In [ ]:
sampledata= np.memmap("train.bin", dtype=np.uint16, mode="r")
print(sampledata[:50])
print(encoder.decode(data[:50]))

# Task 3
**Build the Data Loader — Input/Output Batches**

In [ ]:
batch_size=32
block_size=128


device= "cuda" if torch.cuda.is_available() else "cpu"

traindata= np.memmap("train.bin", dtype=np.uint16,mode="r")
valdata= np.memmap("val.bin", dtype=np.uint16,mode="r")

In [ ]:
def get_batch(split):
    data=traindata if split=="train" else valdata

    k=torch.randint(len(data)-block_size-1,(batch_size,))

    x=[]

    for i in k:
        x.append(torch.from_numpy(data[i:i+block_size].astype(np.int64)))
    x= torch.stack(x)

    y=[]

    for i in k:
        y.append(torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)))
    y= torch.stack(x)

    x=x.to(device)
    y=y.to(device)

    return x,y


In [ ]:
x,y= get_batch("train")

print(x[0])
print(y[0])
print(x.shape)
print(y.shape)

We can see that the x and y are shifted by just one position.

# Task 4
**Design the Transformer Model Architecture**



In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size:int = 50257
    block_size:int = 128
    n_layer:int =  6
    n_head:int =6
    n_embd:int =384
    dropout:float =0.2


config=GPTConfig()


 





In [ ]:
class Head(nn.module):
    def __init__(self,config):
        super().__init__()
        headsize=config.n_embd//config.n_head

        self.key=nn.Linear(config.n_embd,headsize, bias = False)
        self.query=nn.Linear(config.n_embd,headsize, bias = False)
        self.value=nn.Linear(config.n_embd,headsize, bias = False)

        self.register_buffer("tril", torch.tril(torch.ones(config.block_size,config.block_size)))

        self.dropout=nn.Dropout(config.dropout)

    def forward(self,x):
        B,T,C = x.shape

        k=self.key(x)
        q= self.query(x)
        v= self.value(x)

        weight=q@k.transpose(-2,-1)
        weight=weight*(k.size(-1)**-0.5)

        weight=weight.masked_fill(self.tril[:T,:T]==0,float("-inf"))

        weight=F.softmax(weight,dim=-1)
        weight=self.dropout(weight)

        out=weight@v

        return out

In [ ]:
class Multihead(nn.Module):
     def __init__(self,config):
        super().__init__()

        self.heads==nn.ModuleList([Head(config) for _ in range(config.n_head)])

        self.proj=nn.Linear(config.n_embd,config.n_embd)

        self.dropout=nn.Dropout(config.dropout)

        def forward(self,x):

            output= torch.cat([h(x) for h in self.heads], dim=-1)
            output= self.proj(output)
            output=self.dropout(output)

            return output


In [ ]:
class FFN(nn.Module):
     def __init__(self, config):
        super().__init__()

        self.net=nn.Sequential(
            nn.Linear(config.n_embd, 4 *config.n_embd),
            nn.Gelu(),
            nn.Linear(config.n_embd * 4, config.n_embd),
            nn.Dropout(config.dropout)
        )

        def forward(self,x):
            return self.net(x)


In [ ]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.sa = Multihead(config)
        self.ffwd = FFN(config)

        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

In [ ]:
class GPT(nn.Module):
     def __init__(self, config):
        super().__init__()
        self.tokenembeddingtable=nn.Embedding(config.vocab_size,config.n_embd)
        self.positionembedding=nn.Embedding(config.block_size,config.n_embd)

        self.blocks=nn.Sequential(*[Block(config) for _ in range(config.n_layer)])

        self.ln_f=nn.LayerNorm(config.n_embd)

        self.headd=nn.Linear(config.n_embd,config.n_vocab_size)


        def forward(self,index,targets=None):
            B,T= index.shape
            token_embed= self.tokenembeddingtable(index)
            pos_embd=self.positionembedding(torch.arange(T,device=index.device))

            x=token_embed+pos_embd
            x=self.blocks(x)
            x=self.ln_f(x)
            logits= self.headd(x)

            if targets is None:
                loss=none
            else:
                B,T,C= logits.shape

                logits=logits.view(B*T,C)
                targets-targets.view(B*T)

                loss=F.cross_entropy(logits,targets)

            return logits,loss


In [ ]:
model= GPT(config)

idx= torch.randint(0,config.vocab_size,(4,128))

logits, loss = model(idx)
print(logits.shape)
print(loss)

# Task 5 - Set Up the Training Configuration


 **AdamW Optimizer** to update the model parameters.
 **CosineAnnealingLR Scheduler** to gradually reduce the learning rate during training.



In [ ]:
import torch.optim as optim

learning_rate = 3e-4
weight_decay = 0.01

optimizer = optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=1000
)

In [ ]:
print("Initial Learning Rate:", optimizer.param_groups[0]["lr"])

optimizer.zero_grad()
optimizer.step()
scheduler.step()

print("Learning Rate after Scheduler Step:", optimizer.param_groups[0]["lr"])

# Task 6
**Pre-train the SLM**

In [ ]:

import matplotlib.pyplot as plt

max_iters = 5000
eval_interval = 250
eval_iters = 50
learning_rate = 3e-4
best_val_loss = float('inf')

train_losses = []
val_losses = []
val_steps = []

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        val_steps.append(iter)

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), 'best_model.pt')
            print(f"  -> saved new best model (val loss {best_val_loss:.4f})")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(val_steps, train_losses, label='train loss')
plt.plot(val_steps, val_losses, label='val loss')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

# Task 7
**Evaluate the Model**

In [ ]:
checkpoint = torch.load('best_model.pt')
model.load_state_dict(checkpoint)
model.eval()

with torch.no_grad():
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        xb, yb = get_batch('val')
        logits, loss = model(xb, yb)
        losses[k] = loss.item()
    val_loss = losses.mean().item()

perplexity = torch.exp(torch.tensor(val_loss)).item()
print(f"val loss: {val_loss:.4f}")
print(f"perplexity: {perplexity:.2f}")



prompts = [
    "Once upon a time",
    "The weather today",
    "In the future, technology",
    "She opened the door and",
]

for p in prompts:
    context = torch.tensor(encode(p), dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(context, max_new_tokens=100)
    print(decode(out[0].tolist()))
    print('---')